## 5-Day Tempreture Forecast
This noteook uses the processed weather data to tain a multi output model that predicts the next 5 days of temperature.

In [24]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor 
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pickle
import plotly.express as px

In [25]:
df = pd.read_csv('data/features.csv')
df = df.drop(columns=['temp_tomorrow', 'tomorrow_rain', 'date', 'day', 'month'])
df.columns

Index(['cloud_cover', 'sunshine', 'global_radiation', 'max_temp', 'mean_temp',
       'min_temp', 'precipitation', 'pressure', 'snow_depth', 'year',
       'season_spring', 'season_summer', 'season_winter', 'temp_yesterday',
       'rain_yesterday', 'sunshine_yesterday', 'pressure_yesterday',
       'cloud_yesterday', 'temp_last3', 'pressure_last3', 'sun_last3',
       'cloud_last3', 'rain_last3', 'temp_last5', 'pressure_last5',
       'sun_last5', 'cloud_last5', 'rain_last5'],
      dtype='object')

In [26]:
# The 5 target column, one for each future day
df['temp_day1'] = df['mean_temp'].shift(-1)
df['temp_day2'] = df['mean_temp'].shift(-2)
df['temp_day3'] = df['mean_temp'].shift(-3)
df['temp_day4'] = df['mean_temp'].shift(-4)
df['temp_day5'] = df['mean_temp'].shift(-5)

df = df.dropna()

#### Decision Tree for Feature Importance
Again to determine if the importance has shifted from the previous model for day 5.

In [27]:
X = df.drop(columns=['temp_day5', 'temp_day4', 'temp_day3', 'temp_day2', 'temp_day1'])
y = df['temp_day5']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [28]:
tree = DecisionTreeRegressor(random_state=42)
tree.fit(X_train, y_train)

y_test_pred = tree.predict(X_test)

In [29]:
importance = pd.Series(tree.feature_importances_, index=X.columns).sort_values(ascending=False)
importance = importance.reset_index(name="importance")

fig_importance = px.bar(importance, x='index', y='importance', title='Feature Importance')
fig_importance.show()

In [38]:
final_features = ['max_temp', 'season_summer', 'season_winter', 'temp_last5', 'year']

X = df[final_features]
y = df[['temp_day1', 'temp_day2', 'temp_day3', 'temp_day4', 'temp_day5']]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [40]:
model = MultiOutputRegressor(LinearRegression())
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [42]:
results = []
for i, col in enumerate(['Day 1', 'Day 2', 'Day 3', 'Day 4', 'Day 5']):
    mae  = mean_absolute_error(y_test.iloc[:, i], y_pred[:, i])
    r2   = r2_score(y_test.iloc[:, i], y_pred[:, i])
    results.append({'Day': col, 'MAE': round(mae, 3), 'R²': round(r2, 3)})

results_df = pd.DataFrame(results)
print(results_df)


fig = px.line(results_df, x='Day', y=['MAE', 'R²'], markers=True,
              title='Multi-Output Model: Accuracy by Forecast Day')
fig.show()

     Day    MAE     R²
0  Day 1  1.073  0.944
1  Day 2  1.495  0.883
2  Day 3  1.912  0.818
3  Day 4  2.169  0.769
4  Day 5  2.311  0.735


* MAE going up: The further into the future the more off the predictions get. Each extra day adds roughly +0.3C of error.
* R2 going down: The model is less able to explain of the variation in the data as we predict further into the future. Worth noting they are gentle steady sloped, not a cliff, which is good.

Overall the shape of this chart is exactly what a healthy multi output model looks like.

In [ ]:
# Saving model
final_features = ['max_temp', 'season_summer', 'season_winter', 'temp_last5', 'year']

with open("models/multi_temperature_model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("models/multi_temperature_features.pkl", "wb") as f:
    pickle.dump(final_features, f)